**Import extracted data** (continuous outcomes)\
`./data/extracted`

In [1]:
import pandas as pd
df = pd.read_csv(f"./data/continuous.csv", encoding = "utf-8")
globals()[f"continuous"] = df

In [3]:
outcomes = [
    "ikdc_subjective",
    "lysholm",
    "tegner",
    "instrumented_laxity"
]

for x in outcomes:
    import pandas as pd
    df = continuous[continuous["outcome"] == f"{x}"]
    df.to_csv(f"./data/extracted/csv/{x}.csv", encoding = "utf-8")
    df.to_excel(f"./data/extracted/xlsx/{x}.xlsx")
    globals()[x] = df

2. **Pivot the data** (one row per study)\
`./data/reformatted`

In [4]:
dfs = {
    "ikdc_subjective": ikdc_subjective, 
    "lysholm": lysholm, 
    "tegner": tegner, 
    "instrumented_laxity": instrumented_laxity
}

for name, df in dfs.items():
    df = pd.read_csv(f"./data/extracted/csv/{x}.csv", encoding = "utf-8")

    df["group"] = df.groupby(["study"]).cumcount() + 1
    
    # reshape
    wide = (
        df.pivot(
            index= "id",
            columns="group",
            values=["study", "subgroup", "ni", "yi", "sdi", "age", "male", "female", "follow_up"]
        )
    )
    
    # flatten column names
    wide.columns = [
        f"{name}{group}" if name != "subgroup" else f"treat{group}"
        for name, group in wide.columns
    ]
    
    # rename variables
    wide = (
        wide.rename(columns={
            "ni1": "n1",
            "yi1": "m1",
            "sdi1": "sd1",
            "ni2": "n2",
            "yi2": "m2",
            "sdi2": "sd2",
        })
        .reset_index()
    )

    wide["n1"] = wide["n1"].astype(int)
    wide["n2"] = wide["n2"].astype(int)
    wide["sd1"] = wide["sd1"].astype(float)
    wide["sd2"] =wide["sd2"].astype(float)
    wide["m1"] = wide["m1"].astype(float)
    wide["m2"] = wide["m2"].astype(float)
    
    import os
    os.makedirs(f"./data/reformatted/csv", exist_ok = True)
    wide.to_csv(f"./data/reformatted/csv/{name}.csv", encoding = "utf-8")
    
    os.makedirs(f"./data/reformatted/xlsx", exist_ok = True)
    wide.to_excel(f"./data/reformatted/xlsx/{name}.xlsx")
    
    globals()[name] = wide

In [5]:
ikdc_subjective.head()

,id,study1,study2,treat1,treat2,n1,n2,m1,m2,sd1,sd2,age1,age2,male1,male2,female1,female2,follow_up1,follow_up2
0,10,"Barié et al., 2020","Barié et al., 2020",BPTB,QT,43,43,91.0,92.0,7.3,11.5,30.5,30.5,17.0,17.0,13.0,13.0,123.6,123.6
1,12,"Bi et al., 2018","Bi et al., 2018",HT,PLT,62,62,90.4,89.3,7.1,8.4,29.1,29.1,34.0,34.0,28.0,28.0,29.7,29.7
2,13,"Björnsson et al., 2016","Björnsson et al., 2016",BPTB,HT,61,86,67.3,74.0,20.8,18.8,26.8,26.8,53.0,53.0,33.0,33.0,191.9,191.9
3,14,"Bottoni et al., 2015","Bottoni et al., 2015",HT,TA,44,36,79.5,81.1,24.7,23.1,29.2,29.2,43.0,43.0,6.0,6.0,120.0,120.0
4,18,"Butt et al., 2024","Butt et al., 2024",HT,PLT,30,30,89.7,89.9,5.7,9.8,27.7,27.7,30.0,30.0,0.0,0.0,60.0,60.0


3. **Prepare sheets for analysis** (calculate standardized mean differences and standard errors of standardized mean differences as TE and seTE, respectively)\
`./data/sheets_for_analysis`

In [6]:
import numpy as np

dfs = {
    "ikdc_subjective": ikdc_subjective, 
    "lysholm": lysholm, 
    "tegner": tegner, 
    "instrumented_laxity": instrumented_laxity
}

for name, df in dfs.items():
    # Standardized Mean Difference (SMD)
    
    def SMD(n1, n2, m1, m2, sd1, sd2):
        sd = (
            (((n1 - 1) * sd1**2) + ((n2 - 1) * sd2**2))
            / (n1 + n2 - 2)
        )
        sp = np.sqrt(sd)
        return round((np.abs(m1 - m2) / sp), 3)
    
    # Standard Error of the Standardized Mean Difference (SMD)
    def seSMD(n1, n2, smd):
        se = np.sqrt(
            (n1 + n2)/(n1*n2)
            + smd**2/(2*(n1+n2-2))
        )
        se = round(se, 3)
        return se

    df["TE"] = SMD(df["n1"], df["n2"], df["m1"], df["m2"], df["sd1"], df["sd2"])
    df["seTE"] = seSMD(df["n1"], df["n2"], df["TE"])

    df.to_csv(f"./data/sheets_for_analysis/csv/{name}.csv", encoding = "utf-8")
    df.to_excel(f"./data/sheets_for_analysis/xlsx/{name}.xlsx")

    globals()[name] = df

**Display preview dataframes** (to check for correct columns for network meta-analysis)  

In [7]:
ikdc_subjective.head()

,study,treat1,treat2,n1,n2,m1,m2,sd1,sd2,age1,age2,male1,male2,female1,female2,follow_up1,follow_up2,TE,seTE
0,"Barié et al., 2020",BPTB,QT,43,43,91.0,92.0,7.3,11.5,30.5,30.5,17.0,17.0,13.0,13.0,123.6,123.6,0.104,0.216
1,"Bi et al., 2018",HT,PLT,62,62,90.4,89.3,7.1,8.4,29.1,29.1,34.0,34.0,28.0,28.0,29.7,29.7,0.141,0.180
2,"Björnsson et al., 2016",BPTB,HT,61,86,67.3,74.0,20.8,18.8,26.8,26.8,53.0,53.0,33.0,33.0,191.9,191.9,0.341,0.169
3,"Bottoni et al., 2015",HT,TA,44,36,79.5,81.1,24.7,23.1,29.2,29.2,43.0,43.0,6.0,6.0,120.0,120.0,0.067,0.225
4,"Butt et al., 2024",HT,PLT,30,30,89.7,89.9,5.7,9.8,27.7,27.7,30.0,30.0,0.0,0.0,60.0,60.0,0.025,0.258


In [8]:
lysholm.head()

,study,treat1,treat2,n1,n2,m1,m2,sd1,sd2,age1,age2,male1,male2,female1,female2,follow_up1,follow_up2,TE,seTE
0,"Ahldén et al., 2009",BPTB,HT,25,22,80.70,79.16,21.64,13.35,28.6,28.6,18.0,18.0,7.0,7.0,86.8,86.8,0.084,0.292
1,"Barenius et al., 2010",BPTB,HT,78,75,85.82,86.42,11.83,12.53,35.0,35.0,51.0,51.0,24.0,24.0,96.0,96.0,0.049,0.162
2,"Barié et al., 2020",BPTB,QT,43,43,95.20,95.60,6.60,7.80,30.5,30.5,17.0,17.0,13.0,13.0,123.6,123.6,0.055,0.216
3,"Björnsson et al., 2016",BPTB,HT,61,86,79.40,80.70,16.90,15.30,26.8,26.8,53.0,53.0,33.0,33.0,191.9,191.9,0.081,0.167
4,"Butt et al., 2024",HT,PLT,30,30,95.10,94.50,5.20,6.40,27.7,27.7,30.0,30.0,0.0,0.0,60.0,60.0,0.103,0.258


In [9]:
tegner.head()

,study,treat1,treat2,n1,n2,m1,m2,sd1,sd2,age1,age2,male1,male2,female1,female2,follow_up1,follow_up2,TE,seTE
0,"Ageberg et al., 2009",BPTB,HT,16,20,4.17,3.15,1.41,1.34,28.0,28.0,12.0,12.0,4.0,4.0,36.0,36.0,0.744,0.347
1,"Ahldén et al., 2009",BPTB,HT,25,22,4.87,4.86,2.29,1.31,28.6,28.6,18.0,18.0,7.0,7.0,86.8,86.8,0.005,0.292
2,"Barenius et al., 2010",BPTB,HT,78,75,5.80,4.07,1.87,1.88,35.0,35.0,51.0,51.0,24.0,24.0,96.0,96.0,0.923,0.170
3,"Barié et al., 2020",BPTB,QT,43,43,6.10,6.10,1.14,1.14,30.5,30.5,17.0,17.0,13.0,13.0,123.6,123.6,0.000,0.216
4,"Björnsson et al., 2016",BPTB,HT,61,86,4.10,4.00,1.70,1.70,26.8,26.8,53.0,53.0,33.0,33.0,191.9,191.9,0.059,0.167


In [13]:
instrumented_laxity.head()

,index,study,treat1,treat2,n1,n2,m1,m2,sd1,sd2,age1,age2,male1,male2,female1,female2,follow_up1,follow_up2,TE,seTE
0,0,"Aglietti et al., 2004",BPTB,HT,60,60,2.20,1.95,1.08,1.3,25.0,25.0,46.0,46.0,14.0,14.0,24.0,24.0,0.209,0.183
1,1,"Ahldén et al., 2009",BPTB,HT,25,22,2.60,1.40,3.30,2.6,28.6,28.6,18.0,18.0,7.0,7.0,86.8,86.8,0.401,0.295
2,2,"Anderson et al., 2001",BPTB,HT,35,33,2.10,3.10,2.00,2.3,21.0,21.0,45.0,45.0,23.0,23.0,35.8,35.8,0.465,0.246
3,3,"Aune et al., 2001",BPTB,HT,32,29,3.30,2.90,2.60,2.4,27.0,27.0,21.0,21.0,16.0,16.0,24.0,24.0,0.160,0.257
4,4,"Barié et al., 2020",BPTB,QT,43,43,1.05,1.00,1.36,1.1,30.5,30.5,17.0,17.0,13.0,13.0,123.6,123.6,0.040,0.216
